[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnmarktaylor91/torchlens/blob/main/notebooks/learn/09_scale.ipynb)

In [1]:
# Colab setup. Does nothing when this notebook runs from a TorchLens checkout
# (learn_models.py is beside it). On Colab there is no checkout, so this installs
# TorchLens from the main branch and fetches learn_models.py next to the notebook.
import pathlib
import shutil
import subprocess
import sys
import urllib.request

if not pathlib.Path("learn_models.py").exists():
    REPO = "johnmarktaylor91/torchlens"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"torchlens[viz] @ git+https://github.com/{REPO}@main",
            "transformers",
            "ipywidgets",
        ],
        check=True,
    )
    if shutil.which("dot") is None:  # drawings need the Graphviz binary
        subprocess.run(["apt-get", "install", "-qq", "-y", "graphviz"], check=True)
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{REPO}/main/notebooks/learn/learn_models.py",
        "learn_models.py",
    )

# Scale

**Level 4 of 5. Starting point: thousands of batches.**

A default capture keeps every activation, which is right for one forward pass and wrong for a
dataset. This
notebook keeps the one keyword you already know, `save=`, and changes what happens to the
tensors it selects: record only those, stream them to disk as the model runs, reduce them to
statistics and discard them, or aggregate statistics across a whole loader. A budget guards the
case where a capture would not fit.

> **You'll want to know:** [`save=` and predicates](04_keep_what_you_need.ipynb)
> (`tl.func("relu")`, `tl.in_module("blocks")`), [`trace.save` and
> `tl.load`](04_keep_what_you_need.ipynb) (a trace as a directory),
> [`tl.extract_dataset`](04_keep_what_you_need.ipynb) (named layers over a dataset).

In [2]:
import tempfile
import warnings
from pathlib import Path

import torch
from IPython.display import display

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [3]:
from learn_models import (
    tiny_cnn,
    tiny_dataset,
    tiny_mlp,
)  # tiny seeded CPU models; nothing downloads

cnn, image = tiny_cnn()  # six conv blocks, each conv, BatchNorm, ReLU; a (2, 1, 16, 16) batch
model, x = tiny_mlp()  # the two-layer MLP, and a (4, 8) input batch

`tl.record` is `tl.trace` with the same `save=`, minus the `Trace`: it keeps the payloads you
selected plus a light metadata row for every op it saw, and builds the full `Trace` from those
rows only if you ask. Payloads it did not select are gone for good.


In [4]:
rec = tl.record(cnn, image, save=tl.func("relu"))
print(
    type(rec).__name__,
    "|",
    len(rec),
    "records kept of",
    len(rec.raw_metadata()),
    "raw capture events seen",
)
rec_trace = rec.to_trace()  # the full structure, on demand
rec_trace["relu_1_2"].out.shape

Recording | 7 records kept of 82 raw capture events seen


torch.Size([2, 8, 16, 16])

## 1. The same keyword, a cheaper entry point

A `Recording` is a sparse event stream: `records` holds the payloads `save=` kept, in capture
order; `raw_metadata()` is the per-op metadata (shapes, parents, labels) that lets `to_trace()`
rebuild the structure you know, with the final labels, and no tensors beyond the kept ones. A
payload that `save=` did not select was never kept, so reading it raises instead of returning a
stale tensor.


In [5]:
first = rec.records[0]
print(
    "one record:",
    first.ctx.func_name,
    "in",
    first.ctx.address or "the root",
    tuple(first.ram_payload.shape),
    "| passes:",
    rec.num_passes,
)
print("final addresses:", rec.to_trace().layer_labels[:5], "...")
print(rec.summary().splitlines()[0])

one record: relu in the root (2, 8, 16, 16) | passes: 1
final addresses: ['input_1', 'conv2d_1_1', 'relu_1_2', 'conv2d_2_3', 'buffer_1'] ...
Recording(n_passes=1, n_records=7, n_grad_records=0)


## 2. Stream to disk while the forward runs

`storage=tl.to_disk(path)` writes each selected activation as it is produced instead of holding
it in memory; the trace that comes back reads from the directory, and `tl.load` reopens it later.
The directory is temporary here; on a real run it is where your results live.

In [6]:
with tempfile.TemporaryDirectory() as tmp:
    run = Path(tmp) / "run.tlspec"
    on_disk = tl.trace(
        cnn,
        image,
        save=tl.func("relu"),
        storage=tl.to_disk(run, include_custom_attributes=False, include_buffer_values=False),
    )
    print(
        "written:",
        sum(p.stat().st_size for p in run.rglob("*") if p.is_file()) // 1024,
        "KB in",
        run.name,
    )
    print("read through the trace:", tuple(on_disk["relu_1_2"].out.shape))
    loaded = tl.load(run)
    print(
        "reopened with tl.load:",
        len(loaded.ops),
        "ops | relu_1_2:",
        tuple(loaded["relu_1_2"].out.shape),
    )

written: 574 KB in run.tlspec
read through the trace: (2, 8, 16, 16)
reopened with tl.load: 23 ops | relu_1_2: (2, 8, 16, 16)


## 3. Reduce and discard

`save="stats"` keeps no activations. During the forward pass every op's output is folded into a
running summary (count, mean, extremes, zero fraction and a few more), and `tl.stats.site_stats`
returns one summary per op label, pass suffix included. For a ReLU the zero fraction is the share of
activation values that are zero in this batch.

In [7]:
stats_trace = tl.trace(cnn, image, save="stats")
site = tl.stats.site_stats(stats_trace)[
    "relu_1_2:1"
].compute()  # keyed by op label, pass suffix included
print("retained for the whole model:", stats_trace.cost.saved_activation_memory)
print(
    f"relu_1_2: mean {site.mean:.3f}, max {site.finite_max:.3f}, zero fraction {site.num_zero / site.numel:.2f}"
)

retained for the whole model: 5.00 KiB
relu_1_2: mean 0.262, max 2.381, zero fraction 0.45


## 4. Statistics over a whole dataset

`tl.stats.aggregate` maps labels to streaming accumulators and drives them over any iterable of
inputs: a mean, a covariance, whatever the accumulator computes, in memory that does not grow
with the number of examples. Paired accumulators such as `tl.stats.CKA` (linear centered kernel
alignment, a similarity between two representations over the same examples) take two tensors
per batch; the streamed value equals the one-shot value on the stacked activations, so you never
need to hold the dataset.


In [8]:
data = tiny_dataset(64)  # 64 MLP inputs as one (64, 8) tensor
results = tl.stats.aggregate(
    model, data.split(16), {"relu_1_2": tl.stats.Mean(), "linear_2_3": tl.stats.Covariance()}
)
print(
    "mean relu activation:",
    round(results["relu_1_2"], 4),
    "| output covariance:",
    tuple(results["linear_2_3"].shape),
)

mean relu activation: 0.2598 | output covariance: (4, 4)


In [9]:
streamed = tl.stats.CKA()
for batch in data.split(16):
    t = tl.trace(model, batch, save=tl.func("relu") | tl.func("linear"))
    streamed.update(t["relu_1_2"].out, t["linear_2_3"].out)

acts = tl.extract_dataset(model, data, ["relu_1_2", "linear_2_3"], batch_size=16, progress=False)
one_shot = tl.stats.cka(acts["relu_1_2"], acts["linear_2_3"])
print(
    f"CKA(relu, output): streamed {float(streamed.compute()):.6f} | one shot {float(one_shot):.6f}"
)

CKA(relu, output): streamed 0.483001 | one shot 0.483001


## 5. Budgets

Every capture runs under `CaptureOptions(save_budget=...)`. The default `"auto"` allows half of
the memory that is free when the capture starts; a float is a fraction of it, an int is a byte
cap. A capture that would cross the budget stops with a typed error naming the numbers, instead
of filling the machine first.

In [10]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the failed attempt also warns that the model was restored
    try:
        tl.trace(cnn, image, capture=tl.options.CaptureOptions(save_budget=4096))
    except tl.errors.SaveBudgetExceededError as error:
        print(type(error).__name__, "|", str(error).splitlines()[0][:160])

SaveBudgetExceededError | torchlens stopped capture: retained activations crossed the save budget on cpu.


## 6. That was level 4

`tl.record` keeps the payloads `save=` names and a metadata row per op; `storage=tl.to_disk(path)` streams it; `save="stats"`
reduces and discards; `tl.stats.aggregate` and the paired accumulators run over a dataset in
constant memory; `save_budget=` raises before a capture hurts.

**One thing about networks:** a running covariance is all that PCA and linear CKA need, and its
memory grows with the square of the feature width, not with the length of the dataset, so a
stream over a million examples costs the same as one over sixteen. Stimulus-wise RSA is
different: a representational dissimilarity matrix compares individual examples, so it needs
their activations, which is why 10 keeps sixteen items to build one.

**Go deeper:** next is [10, Beyond one forward](10_beyond_one_forward.ipynb). Workflow:
[cog_neuro_extraction](../../examples/workflows/cog_neuro_extraction.ipynb). Reference:
[choosing a fast path](../../docs/guides/fast_paths.md) and
[streaming statistics](../../docs/reference/stats.md).